# DPS crystal prior — Colab runner

This notebook contains **no logic**. It clones the repo, installs deps, mounts Drive
and calls the same CLI scripts you run locally. All code lives in git so the local
CPU test loop stays meaningful — edit there, push, re-run here.

**The cell below holds the run switches.** With the defaults (`TRAIN_PRIOR = False`)
"Run all" does setup, then the five-model experiments. It never retrains the prior.
Every training cell prints `SKIPPED` instead.

**When training (`TRAIN_PRIOR = True`), order matters.** Sections 5–7 are a ~1h cheap
read that surfaces problems before you commit many GPU-hours to sections 9–10.

**Runtime → Change runtime type → GPU** before starting.


In [ ]:
# ======================= RUN SWITCHES: check before "Run all" =======================

# Train the diffusion prior (sections 6-10: quick run, quick eval, synth pretrain,
# real finetune -- about a day of GPU). The prior is already trained and on Drive, so
# this is OFF: every training cell then prints SKIPPED and does nothing, and
# "Run all" goes straight from setup to the experiments in Part 2.
TRAIN_PRIOR = False

# Which prior checkpoint the experiments use.
#   None        -> newest model_ema_*.pt in <MODELS>/real_finetune (MODELS is set in section 3)
#   '/content/drive/MyDrive/.../model_ema_XXXXXX.pt' -> exactly that file. Pin one if you
#                  picked a checkpoint from the prominence / NN-distance window (section 10)
#                  rather than the last one.
PRIOR_CKPT = None

# The ~1 min end-to-end smoke test (section 5). Cheap, and worth keeping on after a
# code change -- it catches a broken environment before hours of DPS.
RUN_SMOKE = True


def skipped(what, switch='TRAIN_PRIOR'):
    print(f'SKIPPED: {what}  ({switch} = False in the first cell)')


import subprocess


def sh(cmd):
    """Run a shell command, streaming its output here, and RAISE if it fails.

    `!cmd` never raises, so under "Run all" a failed probe or a crashed sweep
    would let the next cells carry on into hours of wasted (or wrong) DPS.
    The cells whose failure matters use this instead, so Run all stops there.
    """
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}) -- see output '
                           f'above. Stopping here.\n  {cmd[:200]}')


print(f'TRAIN_PRIOR={TRAIN_PRIOR}  PRIOR_CKPT={PRIOR_CKPT}  RUN_SMOKE={RUN_SMOKE}')


In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
# Everything below needs a GPU. When the GPU quota runs out Colab can hand out
# a CPU runtime instead, and the sweep would then quietly run at CPU speed.
assert torch.cuda.is_available(), (
    'No GPU: Runtime -> Change runtime type -> GPU (or wait for quota).')


## 1. Repo


In [ ]:
import os, glob
REPO   = '/content/diffusion-posterior-sampling'
URL    = 'https://github.com/dawidratynski/diffusion-posterior-sampling.git'
BRANCH = 'scratchpad'   # <- the branch holding this work; change when merged to main

# Through sh() so a failure STOPS Run all instead of running stale code.
# The Colab clone is a throwaway copy that must equal the pushed branch exactly,
# so re-runs reset to it: `pip install -e .` rewrites the tracked
# dps_cyclic.egg-info/, which would otherwise make a later pull fail.
if os.path.exists(REPO):
    sh(f'cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} '
       f'&& git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {URL} {REPO}')
%cd {REPO}

# Confirm you actually have the code you pushed: compare this hash with your
# local `git log --oneline -1`. Colab only sees what is pushed to BRANCH.
sh('git rev-parse --abbrev-ref HEAD && git log --oneline -1')
assert os.path.exists('scripts/train_diffusion.py'), \
    'Wrong branch: scripts/train_diffusion.py is missing.'


## 2. Dependencies

`torch` is deliberately **not** a project dependency (it lives in the `cpu` group of
`pyproject.toml`, used only for local CPU dev), so this installs the project's other
deps and leaves Colab's CUDA torch untouched.


In [ ]:
!pip install -q -e .

# Fail loudly rather than limping on with missing deps: a failed editable
# install is easy to miss in Colab's output and only bites much later.
import importlib
for m in ['numpy', 'scipy', 'skimage', 'PIL', 'yaml', 'matplotlib', 'tqdm']:
    importlib.import_module(m)
print('dependencies OK')


## 3. Data (local disk) and checkpoints (Drive)

**Checkpoints go to Drive** — Colab disconnects and `/content` is lost.
**Data goes to local disk** — training reads thousands of small PNGs per epoch
and mounted-Drive I/O would starve the GPU.

The archive is located by searching for the directory containing `real/train`,
so it works whether or not the zip has a top-level folder inside it.

Set `DRIVE` and `ZIP` to match your Drive layout.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = '/content/drive/MyDrive/Non-movables'   # <- your Drive folder
ZIP    = f'{DRIVE}/dataset.zip'                  # <- your archive
MODELS = f'{DRIVE}/models'      # checkpoints on Drive: sessions die
os.makedirs(MODELS, exist_ok=True)

# ---------------------------------------------------------------------------
# THE ONLY THING TO CHANGE WHEN UVCGAN WEIGHTS ARRIVE.
# Two UVCGANs, differing only in adversarial loss (standard GAN vs Wasserstein
# GAN). Each gates its own pair of models independently, so whichever arrives
# first can run alone; every cell below adapts and completed work is kept.
#
# WHAT TO UPLOAD: the uvcgan2 MODEL directory -- the one containing config.json,
# not the training OUTDIR above it. train_crystals.py writes it as
#   <outdir>/crystal_experiments/experiment_1/model_m(uvcgan2)_d(basic)_g(vit-modnet)_<label>/
# Only these 5 files are needed (~0.5 GB instead of ~1.1 GB per model; the rest
# is optimiser and discriminator state):
#   config.json  net_gen_ab.pth  net_gen_ba.pth  net_avg_gen_ab.pth  net_avg_gen_ba.pth
# Rename the folder on Drive to the names below. The EMA generators (avg_gen)
# are used, as uvcgan2's own inference does. If a run stopped early and there
# are no final net_*.pth files, upload checkpoints/ instead: the newest
# checkpoint is then loaded (the probe in section 12 says which one it used).
UVCGAN_GAN_PATH  = f'{DRIVE}/uvcgan_gan'
UVCGAN_WGAN_PATH = f'{DRIVE}/uvcgan_wgan'
HAVE_GAN  = os.path.exists(UVCGAN_GAN_PATH)
HAVE_WGAN = os.path.exists(UVCGAN_WGAN_PATH)
HAVE_UVCGAN = HAVE_GAN or HAVE_WGAN

# Round-trip measurement builders, y = G_RS(real). One full round trip runs per
# builder, each held FIXED across all five models (like-for-like), into
# roundtrip_<builder>/. Running BOTH UVCGANs as builders removes the bias any
# single builder has toward its own pair. Before any weights exist, the analytic
# operator builds it instead. (The scale sweep runs on synth inputs and needs
# no builder.)
RS_MODELS = ([m for m, have in [('uvcgan_gan', HAVE_GAN),
                                ('uvcgan_wgan', HAVE_WGAN)] if have]
             or ['spectral'])

# Experiment outputs live on DRIVE, not /content: the sweep (~8 h) and the final
# runs (~14.5 h) both outlast a Colab session, and resuming only works if the
# finished part survives the disconnect. Section 14 reads the sweep's chosen
# scales from SWEEP_DIR.
SWEEP_DIR = f'{DRIVE}/results_sweep'
FINAL_DIR = f'{DRIVE}/results_final'

print('GAN  UVCGAN weights:', 'FOUND' if HAVE_GAN  else f'not yet at {UVCGAN_GAN_PATH}')
print('WGAN UVCGAN weights:', 'FOUND' if HAVE_WGAN else f'not yet at {UVCGAN_WGAN_PATH}')
print('round trips built by:', RS_MODELS)
# Passed to run_comparison.py; an empty value means "skip those models".
UVCGAN_ARGS = (f'--uvcgan_gan_path "{UVCGAN_GAN_PATH if HAVE_GAN else ""}" '
               f'--uvcgan_wgan_path "{UVCGAN_WGAN_PATH if HAVE_WGAN else ""}"')
RS_MODELS_ARG = ','.join(RS_MODELS)
# ---------------------------------------------------------------------------


def latest_ckpt(d):
    """Newest model_ema_*.pt in d.

    Training cells save every --save_every steps, so you can interrupt one
    at any point and still have a usable checkpoint; this picks it up.
    """
    c = sorted(glob.glob(f'{d}/model_ema_*.pt'))
    if not c:
        raise FileNotFoundError(
            f'No model_ema_*.pt in {d}. Let the training cell reach at least one --save_every interval before running this.')
    return c[-1]


def prior_ckpt():
    """The trained diffusion prior the experiments sample from.

    PRIOR_CKPT from the first cell if set, else the newest finetune checkpoint.
    """
    if PRIOR_CKPT:
        assert os.path.exists(PRIOR_CKPT), f'PRIOR_CKPT {PRIOR_CKPT} does not exist'
        return PRIOR_CKPT
    return latest_ckpt(f'{MODELS}/real_finetune')


# Fail NOW, not after setup, if the prior is missing -- with training skipped
# there is nothing that would produce it.
if not TRAIN_PRIOR:
    print('prior  =', prior_ckpt())

# Data goes on LOCAL disk, not Drive: training reads ~11.6k small PNGs per
# epoch and mounted-Drive I/O on many small files starves the GPU.
EXTRACT = '/content/data_raw'
if not os.path.exists(EXTRACT):
    # Unzip to a temp dir and rename on success: an interrupted unzip must not
    # leave a partial dataset that a re-run would then treat as complete.
    sh(f"rm -rf {EXTRACT}.tmp && mkdir -p {EXTRACT}.tmp "
       f"&& unzip -q '{ZIP}' -d {EXTRACT}.tmp && mv {EXTRACT}.tmp {EXTRACT}")

# Archives may or may not carry a top-level folder, so find the directory
# that actually contains real/train rather than assuming the depth.
matches = sorted(glob.glob(f'{EXTRACT}/**/real/train', recursive=True))
assert matches, f'no */real/train found under {EXTRACT}; check the archive'
DATA = os.path.dirname(os.path.dirname(matches[0]))
print('DATA   =', DATA)
print('MODELS =', MODELS)


def drive_usage():
    """Checkpoints are the only thing that grows. Watch this between stages."""
    !du -sh {MODELS}/* 2>/dev/null; df -h /content/drive | tail -1

!ls {DATA}/real/train | wc -l && ls {DATA}/synth/train | wc -l


## 4. Split hygiene — **run this, do not just dry-run it**

Several crops come from each source photo, so the split must be at *source*
level; splitting on filenames leaks siblings across train/val and makes the
memorisation check meaningless.

**The archive on Drive predates the fix**, so the copy you just unpacked still
leaks. Run the first cell to confirm, then the second to repair it. Idempotent
— safe to re-run, and it skips domains that are already clean.


In [ ]:
sh(f'python scripts/fix_split.py {DATA}/real {DATA}/synth --dry_run')


In [ ]:
# Apply it (expect ~4.6k real files to move; synth is already clean).
# Deterministic (verified: two fresh unzips give identical splits), so every
# session sees the same val/train images and cached results stay valid.
sh(f'python scripts/fix_split.py {DATA}/real {DATA}/synth')


## 5. Environment smoke (~1 min)

The full train → generate → evaluate path at tiny scale. Proves the environment,
data layout and configs work together before anything expensive. Takes ~6 min on
CPU locally; much less here.

It also asserts the exported EMA checkpoint beats chance — guarding a bug where the
training log looked healthy while the saved model was still random initialisation.


In [ ]:
if not RUN_SMOKE:
    skipped('section 5 smoke test', 'RUN_SMOKE')
else:
    sh(f'PY=python ./scripts/smoke_e2e.sh {DATA} /content/smoke')


## 6. Short real-only run (~1h) — the cheap read

A first prior at the **real** 160px resolution, trained only long enough to produce
meaningful output. This is deliberately *not* the final model: it exists to answer
"does DPS through this operator do anything sensible" in an hour rather than a day.

Checkpoints land every 1000 steps, so **you can interrupt this cell whenever you
like** and still use the latest one. Watch the `it/s` in the log to judge how far to
let it go.


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 6 quick real-only training')
else:
    !python scripts/train_diffusion.py \
        --model_config configs/crystal_model_config.yaml \
        --data_root {DATA}/real/train \
        --val_root {DATA}/real/val --val_every 500 \
        --out_dir {MODELS}/quick_real \
        --batch_size 16 --lr 1e-4 --train_steps 8000 \
        --save_every 1000 --amp --resume


## 7. Generate + evaluate from the quick run

`--input_mode roundtrip` builds the measurement as `y = G_RS(real image)` and
records the real image as ground truth. Here the **analytic** operator plays
both roles — it builds `y` and DPS inverts it — which is self-consistent and
needs no trained weights.

Those two roles are separate flags (`--rs_framework`, `--dps_framework`) because
Part 2 needs them to differ: every model there must see the *same* measurement
while inverting its own operator.


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 7 timing on the quick prior')
else:
    CKPT = latest_ckpt(f'{MODELS}/quick_real'); print('using', CKPT)
    !sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml

    # DPS costs a UNet forward AND backward per step, so 1000 steps x many
    # samples runs to hours. Use 100 steps for the cheap read; it is enough to
    # judge whether the output is sane. Part 2 uses a 250-step schedule.
    !sed 's|^timestep_respacing:.*|timestep_respacing: 100|' \
        configs/crystal_diffusion_config.yaml > /content/diffusion_fast.yaml

    # Time ONE sample before committing to a batch. The analytic operator is used
    # here in BOTH roles (it builds the measurement and DPS inverts it), which is
    # self-consistent and needs no trained weights -- see the docstring of
    # scripts/generate_augmented.py on why those two roles are separate flags.
    import time; t0 = time.time()
    !python scripts/generate_augmented.py \
        --model_config /content/model_cfg.yaml \
        --diffusion_config /content/diffusion_fast.yaml \
        --task_config configs/crystal_cyclegan_config.yaml \
        --input_mode roundtrip --real_root {DATA}/real/val \
        --rs_framework spectral \
        --method dps --dps_framework spectral --label dps_analytic \
        --out_dir /content/results/timing \
        --samples_per_input 1 --limit 1
    per_sample = time.time() - t0
    print(f'\n~{per_sample:.0f}s per 100-step sample.')
    print(f'32 refs x 4 variants  = {128*per_sample/60:.0f} min at 100 steps')
    print(f'at the 250-step schedule Part 2 uses = {per_sample*2.5:.0f}s each')


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 7 generation from the quick prior')
else:
    # Adjust --limit from the timing above if needed.
    !python scripts/generate_augmented.py \
        --model_config /content/model_cfg.yaml \
        --diffusion_config /content/diffusion_fast.yaml \
        --task_config configs/crystal_cyclegan_config.yaml \
        --input_mode roundtrip --real_root {DATA}/real/val \
        --rs_framework spectral \
        --method dps --dps_framework spectral --label dps_analytic \
        --out_dir /content/results/quick \
        --samples_per_input 4 --limit 32


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 7 evaluation of the quick prior')
else:
    !python scripts/evaluate.py \
        --result_dir /content/results/quick \
        --real_root {DATA}/real/val \
        --real_train_root {DATA}/real/train \
        --out_csv /content/results/quick_per_image.csv


### Look at the images, not only the table

The metrics cannot tell you whether the output is plausible to a human eye. Compare
a generated image against the reference it was derived from.


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 7 quick-prior image check')
else:
    import matplotlib.pyplot as plt, csv
    rows = list(csv.DictReader(open('/content/results/quick/manifest.csv')))[:4]
    fig, ax = plt.subplots(2, len(rows), figsize=(3*len(rows), 6))
    for i, r in enumerate(rows):
        ax[0, i].imshow(plt.imread(r['source_image'])); ax[0, i].set_title('reference', fontsize=9)
        ax[1, i].imshow(plt.imread(f"/content/results/quick/generated/{r['output_image']}"))
        ax[1, i].set_title('DPS', fontsize=9)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.show()


## 8. `scale` sweep on the quick prior (~75 min) — optional

Superseded by **section 13**, which sweeps every DPS variant against the *final*
prior and at the schedule actually used for results. Run this one only if you
want an early read while sections 9–10 are still pending; otherwise skip
straight to Part 2.

Note the optimum depends on both the prior and the step count, so a value tuned
here does not transfer to the final runs.


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 8 scale sweep on the quick prior')
else:
    SCALES = [0.3, 1.0, 2.0, 3.0]
    N_REFS, N_VARIANTS = 24, 2

    DISCARD_OLD = True   # earlier runs used a different operator; do not reuse them
    if DISCARD_OLD:
        !rm -rf /content/results/scale_*

    print(f'{len(SCALES)} scales x {N_REFS*N_VARIANTS} samples x 100 steps')
    print(f'at ~23 s/sample that is ~{len(SCALES)*N_REFS*N_VARIANTS*23/60:.0f} min total')

    for s in SCALES:
        out = f'/content/results/scale_{s}'
        if os.path.exists(f'{out}/manifest.csv'):
            print(f'scale {s}: already present, skipping')
            continue
        !python scripts/generate_augmented.py \
            --model_config /content/model_cfg.yaml \
            --diffusion_config /content/diffusion_fast.yaml \
            --task_config configs/crystal_cyclegan_config.yaml \
            --input_mode roundtrip --real_root {DATA}/real/val \
            --rs_framework spectral \
            --method dps --dps_framework spectral --label dps_analytic \
            --out_dir {out} --scale {s} \
            --samples_per_input {N_VARIANTS} --limit {N_REFS}

    # Confirm the subsets really are spread across photos, not crops of one scene.
    import csv, re
    for s in SCALES:
        rows = list(csv.DictReader(open(f'/content/results/scale_{s}/manifest.csv')))
        photos = {re.sub(r'_sample_\d+\.png$', '', os.path.basename(r['source_image']))
                  for r in rows}
        print(f'scale {s}: {len(rows)} images from {len(photos)} distinct photos')

    dirs = ' '.join(f'--result_dir /content/results/scale_{s}' for s in SCALES)
    !python scripts/evaluate.py {dirs} --real_root {DATA}/real/val


---
# Only continue once sections 5–7 look right.
Everything below is measured in GPU-hours.
---


## 9. Stage 1 — pretrain on SYNTH (~18 h, 2 sessions)

Synth is effectively unlimited, so this stage carries no memorisation risk and
teaches lattice structure and low-level statistics.

**40k steps, not the 150k originally planned.** At the measured 0.61 it/s,
150k + 40k would be ~87 hours — seven Colab sessions. Synth images are also
clean and highly structured, so they converge far faster than a natural-image
corpus; 40k is a realistic starting budget rather than a target.

**Judge it by the probe, not the step count.** Watch `val @ step ... t=25`:
when it stops falling, extend or stop accordingly. The training loss will have
flattened thousands of steps earlier and tells you nothing.

`--resume` means re-running this cell after a disconnect continues rather than
restarting, so spanning two sessions costs nothing.


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 9 synth pretrain (~18 h)')
else:
    !python scripts/train_diffusion.py \
        --model_config configs/crystal_model_config.yaml \
        --data_root {DATA}/synth/train \
        --val_root {DATA}/synth/val --val_every 2000 \
        --out_dir {MODELS}/synth_pretrain \
        --batch_size 16 --lr 1e-4 --train_steps 40000 \
        --save_every 2000 --keep_last 1 --save_fp16 --amp --resume


### 9b. Free the pretrain optimiser state before section 10

`ckpt_latest.pt` is ~1.35 GB (model + EMA + both AdamW moments) and exists only
to resume *that* run. Section 10 starts a fresh run from the exported weights via
`--init_from`, so once section 9 is finished this file is dead weight — and on a
4 GB Drive it is the difference between fitting and not.

**Only run this once section 9 has actually finished.** Deleting it means section 9
can no longer be extended; you would have to restart it.


In [ ]:
SYNTH_DONE = False   # <- set True only after section 9 reports 'Training finished.'

if not TRAIN_PRIOR:
    skipped('section 9b freeing pretrain optimiser state')
else:
    drive_usage()
    if SYNTH_DONE:
        !rm -f {MODELS}/synth_pretrain/ckpt_latest.pt
        print('freed ~1.35 GB')
        drive_usage()
    else:
        print('set SYNTH_DONE = True once section 9 has finished')


## 10. Stage 2 — finetune on REAL (~4.5 h)

`--init_from` starts from the pretrained weights with a fresh optimiser and
step counter (unlike `--resume`, which continues a run). Lower LR, because the
point is to move the prior to the real domain, not retrain it.

**You now have two stopping signals, and they point in opposite directions:**

| signal | says | source |
|---|---|---|
| `val t=25` still falling | keep training | the probe, logged here |
| `NN dist to real train` approaching 1.11 | stop, it is copying | section 11 |
| `peak prominence` still above real | keep training | section 11 |

Real is the scarce side (~2.3k source photos), so this stage is where
memorisation happens. Re-run section 11 against successive checkpoints. Stop
where prominence has come down to the real value but NN distance has not yet
fallen to the floor.

If no such window exists — prominence never reaches real before NN distance
collapses — that is a genuine finding about the real set being too small, not a
failure. Report it as such.


In [ ]:
if not TRAIN_PRIOR:
    skipped('section 10 real finetune (~4.5 h)')
else:
    PRETRAINED = latest_ckpt(f'{MODELS}/synth_pretrain'); print('init from', PRETRAINED)
    !python scripts/train_diffusion.py \
        --model_config configs/crystal_model_config.yaml \
        --data_root {DATA}/real/train \
        --val_root {DATA}/real/val --val_every 1000 \
        --out_dir {MODELS}/real_finetune \
        --init_from {PRETRAINED} \
        --batch_size 16 --lr 2e-5 --train_steps 10000 \
        --save_every 1250 --keep_last 8 --save_fp16 --amp --resume


---
# Part 2 — the five-model comparison

Everything below produces the report. It compares five synth→real models:

| model | what it is | needs weights |
|---|---|---|
| `uvcgan_gan` | direct S→R translation, GAN-trained UVCGAN | GAN |
| `uvcgan_wgan` | direct S→R translation, WGAN-trained UVCGAN | WGAN |
| `dps_uvcgan_gan` | DPS inverting the GAN UVCGAN's R→S generator | GAN |
| `dps_uvcgan_wgan` | DPS inverting the WGAN UVCGAN's R→S generator | WGAN |
| `dps_analytic` | DPS inverting the **analytic** operator | none |

**These cells need no editing when weights arrive.** Set `UVCGAN_GAN_PATH` /
`UVCGAN_WGAN_PATH` in section 3, re-run that cell, then re-run these. Each
UVCGAN gates its own pair of models, so they can arrive at different times; work
already done is kept. With neither, `dps_analytic` runs alone, which still
validates the pipeline, metrics and figures end to end.

The experiments, each run for every model:

- **`roundtrip_uvcgan_gan`, `roundtrip_uvcgan_wgan`**: real → `G_RS` → model,
  once with each UVCGAN's `G_RS` building the measurement. Within one round trip
  all five models get the identical `y`. The original real image is ground
  truth, so PSNR/SSIM/LPIPS apply.
- **`synth`**: synthetic image → model. The actual deliverable. No ground truth
  (given a synthetic input there is no single correct realistic image), so
  distributional and lattice metrics only.

**Why two round trips.** A round trip structurally favours the UVCGAN whose
`gen_ba` built the measurement, since its `gen_ab` was trained with a cycle loss
to invert exactly that generator. Its DPS sibling also inverts the exact
operator that produced `y`. Under a single builder, GAN vs WGAN is therefore
confounded with "was the builder". With both, each UVCGAN is measured once on
home ground and once away, and the comparison is symmetric. `dps_analytic` is
mismatched in both, which is what that ablation measures. The synth
experiment has no builder and no such bias.


## 12. Probe the UVCGAN checkpoints — run FIRST when weights arrive

`load_uvcgan2` is verified against the uvcgan2 library on model directories
built with `train_crystals.py`'s exact config (GAN and WGAN), but never on
*trained* weights. The probe closes that gap in seconds. For each model it
reports what is in the directory and which weights it loaded (EMA generator;
final or checkpoint), then walks the exact path the experiments take:

- the raw generator really is a [0, 1] → [0, 1] model (uvcgan2 trains on
  un-normalised images and ends in a sigmoid; the loader converts to this
  pipeline's [−1, 1], and this checks that assumption holds);
- shapes, ranges and determinism;
- the DPS gradient survives;
- `G_RS` actually lands in the synth domain. This catches a reversed direction
  convention (is domain `a` synth in your uvcgan2 config?) or an untrained/early
  checkpoint. A generator can load, run and differentiate perfectly while
  mapping the wrong way.

Every check counts toward the final PASS/FAIL. **Don't start section 13 on a
FAIL.**


In [ ]:
# Install uvcgan2 itself. The loader needs its code to rebuild the generator
# architecture from the saved config -- the checkpoint alone is not enough.
#
# Pinned to the commit load_uvcgan2 was verified against, so an upstream change
# cannot silently break loading mid-thesis. If the UVCGANs were trained with a
# different uvcgan2 commit and loading fails, set it to that commit.
UVCGAN2_COMMIT = '871b2b9'
#
# --no-deps keeps pip from touching Colab's CUDA torch. uvcgan2 only needs
# numpy, pandas, tqdm, Pillow and torch/torchvision, all already on Colab.

if not HAVE_UVCGAN:
    print('No weights yet -- skipping uvcgan2 install.')
else:
    UVCGAN_SRC = '/content/uvcgan2'
    if not os.path.exists(UVCGAN_SRC):
        sh(f'git clone -q https://github.com/LS4GAN/uvcgan2.git {UVCGAN_SRC}')
    sh(f'cd {UVCGAN_SRC} && git fetch -q && git checkout -q {UVCGAN2_COMMIT} '
       f'&& git log --oneline -1')
    sh(f'pip install -q -e {UVCGAN_SRC} --no-deps')
    # Checked in a FRESH process: an editable install made by a subprocess is
    # not importable in this already-running kernel, but every script below
    # runs in a new process, which is what matters.
    sh('python -c "import uvcgan2.cgan, uvcgan2.config, torch; '
       'assert torch.cuda.is_available(), \'torch lost CUDA\'; '
       'print(\'uvcgan2 OK | torch\', torch.__version__)"')


In [ ]:
# === 12. Probe the UVCGAN checkpoints — run FIRST when weights arrive =======
# See the markdown above. Raises on any FAIL, so "Run all" stops here instead
# of starting hours of DPS on an operator that is wired wrong.

if not HAVE_UVCGAN:
    print('No UVCGAN weights yet.\n'
          'Set UVCGAN_GAN_PATH / UVCGAN_WGAN_PATH in section 3 when they arrive, '
          're-run that cell, then this one.\n'
          'Everything below still runs with dps_analytic alone.')
for name, have, path in [('GAN', HAVE_GAN, UVCGAN_GAN_PATH),
                         ('WGAN', HAVE_WGAN, UVCGAN_WGAN_PATH)]:
    if not have:
        continue
    print(f'\n################ {name} UVCGAN ################')
    sh(f'python scripts/probe_uvcgan.py --path "{path}" '
       f'--real_root {DATA}/real/val --synth_root {DATA}/synth/val')


## 13. Hyperparameter sweep — conditioning scale, per DPS variant

Each DPS variant inverts a **different operator**, so each needs its own scale:
ζ multiplies the gradient of ‖y − A(x̂₀)‖, whose size depends on A. To keep the
three-way comparison fair, all three are tuned **by the same procedure, on the
same budget, chosen by the same fixed rule**. Nothing here is picked by eye.

**Procedure** (`run_comparison.py --stage sweep`):

1. **Coarse log grid** `0.1, 0.3, 1, 3, 10`. Wide on purpose: the trained
   generators' gradients may differ from the analytic operator's by an order of
   magnitude.
2. **Refinement:** the geometric midpoints around the coarse winner, or one step
   past the edge if the winner sits at one.
3. **Selection rule**, over both rounds together:
   - *admissible* = lattice preserved (`signature_err_median ≤ 0.05`) **and** not
     memorising (`nn_dist_median ≥ 0.98 × floor`);
   - among those, the lowest **KID** (realism);
   - KID differences within the best scale's subset spread are noise, so among
     those the best **label validity** wins.

**Inputs are `synth/train` images**, i.e. the deliverable's own input type, and
disjoint (down to the source photo) from the `synth/val` inputs of the final
runs. So the scale is not tuned on the images it is then scored on. No round
trip is involved, so no measurement builder has to be chosen. Every metric the
rule uses is unpaired.

**Output** (on Drive, `results_sweep/`): `chosen_scales.json` (scale, reason,
warnings per model) plus one full metric table per model, `<model>_sweep.csv`,
for the report's appendix. Read the warnings: a choice at the grid edge, a
collapse in diversity, or no admissible scale at all. Each needs a look before
section 14.

**Budget:** 7 scales × 24 inputs × 2 variants ≈ 2.7 h per DPS model at ~29 s per
sample, so ≈ 8 h for all three. Resumable, and a model already swept is not
redone, so it is fine to run it as each UVCGAN arrives.


In [ ]:
SWEEP_SCALES = '0.1,0.3,1,3,10'   # coarse grid; refinement is added automatically
SWEEP_REFS, SWEEP_VARIANTS = 24, 2

CKPT = prior_ckpt(); print('prior:', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml

# One schedule shared by the sweep and section 14, so the tuned scale is the
# scale that actually gets used. DPS applies one guidance step per timestep, so
# total guidance scales with schedule length: a value tuned at 100 steps
# over-guides at 250.
!sed 's|^timestep_respacing:.*|timestep_respacing: 250|' \
    configs/crystal_diffusion_config.yaml > /content/diffusion_eval.yaml

n_dps = 1 + HAVE_GAN + HAVE_WGAN
n_scales = len(SWEEP_SCALES.split(',')) + 2   # + refinement
print(f'~{n_dps * n_scales * SWEEP_REFS * SWEEP_VARIANTS * 29 / 3600:.1f} h '
      f'at ~29 s/sample ({n_dps} DPS variant(s) x ~{n_scales} scales)')

# Already-finished runs are skipped, so when a UVCGAN arrives later re-running
# this only sweeps the new model; earlier choices in chosen_scales.json are kept.
# SWEEP_DIR (section 3) is on Drive, unlike other results: section 14 depends on
# its output, and a dead session should not cost the ~8 h it took.
# sh() raises on failure, so section 14 never runs on a half-finished sweep.
sh(f'python scripts/run_comparison.py --stage sweep '
   f'--data_root {DATA} '
   f'--model_config /content/model_cfg.yaml '
   f'--diffusion_config /content/diffusion_eval.yaml '
   f'--task_config configs/crystal_cyclegan_config.yaml '
   f'--out_dir {SWEEP_DIR} '
   f'{UVCGAN_ARGS} '
   f'--scales {SWEEP_SCALES} '
   f'--sweep_limit {SWEEP_REFS} --sweep_samples {SWEEP_VARIANTS}')


In [ ]:
# The sweep tables (report appendix) and the rule's choice for each model.
import glob, json

import pandas as pd

COLS = ['scale', 'signature_err_median', 'spacing_err_median',
        'spacing_err_strong', 'nn_dist_median', 'nn_floor', 'kid', 'kid_std',
        'prominence_median', 'real_prominence_median', 'spectrum_dist_to_real',
        'diversity_rmse']

chosen_file = f'{SWEEP_DIR}/chosen_scales.json'
chosen = json.load(open(chosen_file)) if os.path.exists(chosen_file) else {}

for f in sorted(glob.glob(f'{SWEEP_DIR}/*_sweep.csv')):
    label = os.path.basename(f).removesuffix('_sweep.csv')
    df = pd.read_csv(f).sort_values('scale')
    print(f'\n=== {label} ===')
    print(df[[c for c in COLS if c in df.columns]].to_string(index=False))
    pick = chosen.get(label)
    if pick:
        print(f'-> chosen scale {pick["scale"]:g}: {pick["reason"]}')
        for w in pick.get('warnings', []):
            print(f'   WARNING: {w}')


In [ ]:
# Scales used by section 14: taken from the sweep's rule, per model.
#
# This cell REFUSES to continue (so "Run all" stops before ~14.5 h of DPS) when
# a model that has weights:
#   - has no sweep result yet (would silently use a fallback default),
#   - was swept with a different prior than the one now in use (stale pick),
#   - has a sweep warning (no admissible scale, optimum at the grid edge,
#     collapsed diversity) -- each means "look before spending GPU on it".
# To proceed anyway, make the decision explicit:
#   OVERRIDE = {'dps_uvcgan_gan': 1.5}   # use this scale (bypasses the rule;
#                                         # say so in the report)
#   ACCEPT   = {'dps_analytic'}          # use the rule's pick despite warnings
DEFAULTS = {'dps_analytic': 2.0, 'dps_uvcgan_gan': 1.0, 'dps_uvcgan_wgan': 1.0}
OVERRIDE = {}
ACCEPT = set()

import json
chosen_file = f'{SWEEP_DIR}/chosen_scales.json'
chosen = json.load(open(chosen_file)) if os.path.exists(chosen_file) else {}
active = {'dps_analytic', *(['dps_uvcgan_gan'] if HAVE_GAN else []),
          *(['dps_uvcgan_wgan'] if HAVE_WGAN else [])}
prior_now = prior_ckpt()

SCALES, problems = {}, []
for label, default in DEFAULTS.items():
    pick = chosen.get(label)
    if label in OVERRIDE:
        SCALES[label], src = OVERRIDE[label], 'OVERRIDE (bypasses the rule)'
    elif pick:
        SCALES[label], src = pick['scale'], 'sweep rule'
        if label in active and label not in ACCEPT:
            if pick.get('prior_model_path') not in (None, prior_now):
                problems.append(f'{label}: swept with prior '
                                f'{pick["prior_model_path"]}, now {prior_now} '
                                '-- re-run section 13')
            for w in pick.get('warnings', []):
                problems.append(f'{label}: {w}')
    else:
        SCALES[label], src = default, 'FALLBACK default -- not swept'
        if label in active:
            problems.append(f'{label}: not swept yet (run section 13)')
    print(f'{label:<16} {SCALES[label]:g}   [{src}]'
          + ('' if label in active else '   (no weights; not run)'))

if problems:
    raise RuntimeError('Not starting section 14:\n  ' + '\n  '.join(problems)
                       + '\nResolve, or set OVERRIDE / ACCEPT above explicitly.')

SCALE_DPS_ANALYTIC    = SCALES['dps_analytic']
SCALE_DPS_UVCGAN_GAN  = SCALES['dps_uvcgan_gan']
SCALE_DPS_UVCGAN_WGAN = SCALES['dps_uvcgan_wgan']


## 14. Final comparison — every experiment, all available models

Generates, evaluates and builds the figures in one pass: the **synth**
experiment first (the deliverable, so it is the first thing to finish), then
one round trip **per builder in `RS_MODELS`** (both UVCGANs once their weights
exist). Resumable: a re-run after weights arrive redoes only what is missing.
Results for an earlier builder set are kept alongside.

**Budget.** Each DPS model costs `limit × samples × 250 steps`. At 50 references
× 4 variants that is 200 samples ≈ 1.6 h per DPS model per experiment. The two
direct `uvcgan_*` models are a single forward pass each and essentially free.
Lower `FINAL_LIMIT` if units are tight; the metrics stabilise well before 50
references.

| weights present | experiments | DPS models | ≈ time |
|---|---|---|---|
| both | synth + 2 round trips | 3 | ≈ 14.5 h |
| one | synth + 1 round trip | 2 | ≈ 6.5 h |
| none | synth + 1 round trip (analytic builder) | 1 | ≈ 3.2 h |

With both weights this spans several sessions. That's fine: re-run the cell and it
continues where it stopped.


In [ ]:
FINAL_LIMIT, FINAL_SAMPLES = 50, 4

# Rebuilt here rather than inherited from section 13, so this cell also works if
# you skipped the sweep and reused previously chosen scales. Idempotent.
CKPT = prior_ckpt(); print('prior:', CKPT)
!sed 's|^model_path:.*|model_path: {CKPT}|' configs/crystal_model_config.yaml > /content/model_cfg.yaml
!sed 's|^timestep_respacing:.*|timestep_respacing: 250|' \
    configs/crystal_diffusion_config.yaml > /content/diffusion_eval.yaml

n_dps = 1 + HAVE_GAN + HAVE_WGAN
n_exp = len(RS_MODELS_ARG.split(',')) + 1
print(f'~{n_exp * n_dps * FINAL_LIMIT * FINAL_SAMPLES * 29 / 3600:.1f} h '
      f'for {n_dps} DPS model(s) x {n_exp} experiments (less if partly done)')
print('round trips built by:', RS_MODELS_ARG)

# FINAL_DIR is on Drive (section 3): after a disconnect, re-running this cell
# continues from the last finished model instead of starting over.
sh(f'python scripts/run_comparison.py --stage final '
   f'--data_root {DATA} '
   f'--model_config /content/model_cfg.yaml '
   f'--diffusion_config /content/diffusion_eval.yaml '
   f'--task_config configs/crystal_cyclegan_config.yaml '
   f'--out_dir {FINAL_DIR} '
   f'{UVCGAN_ARGS} '
   f'--rs_models {RS_MODELS_ARG} '
   f'--scale_dps_analytic {SCALE_DPS_ANALYTIC} '
   f'--scale_dps_uvcgan_gan {SCALE_DPS_UVCGAN_GAN} '
   f'--scale_dps_uvcgan_wgan {SCALE_DPS_UVCGAN_WGAN} '
   f'--limit {FINAL_LIMIT} --samples {FINAL_SAMPLES}')


In [ ]:
# The comparison tables the report is built on: the synth experiment plus one
# per round-trip builder that has been run.
import glob

import pandas as pd

COLS = ['model', 'scale', 'spacing_err_median', 'spacing_err_strong',
        'spacing_err_p90', 'signature_err_median', 'angle_err_median',
        'psnr_median', 'ssim_median', 'lpips_median',
        'prominence_median', 'src_prominence_median', 'spectrum_dist_to_real',
        'kid', 'kid_std', 'fid', 'fid_reliable',
        'diversity_rmse', 'nn_dist_median']

tables = sorted(glob.glob(f'{FINAL_DIR}/*_metrics.csv'))
if not tables:
    print('nothing run yet')
for f in tables:
    df = pd.read_csv(f)
    print(f'\n=== {os.path.basename(f).removesuffix("_metrics.csv")} ===')
    print(df[[c for c in COLS if c in df.columns]].to_string(index=False))


In [ ]:
# Look at the figures before trusting the tables. Aggregates cannot tell you
# whether the output is plausible to a human eye; twice in this project the eye
# caught something the metrics only articulated afterwards.
import glob

from IPython.display import Image as ShowImage
from IPython.display import display

for f in sorted(glob.glob(f'{FINAL_DIR}/figures/*/grids/comparison_01.png')):
    exp = f.split('/')[-3]
    print(f'=== {exp}: reference | measurement | each model ===')
    display(ShowImage(filename=f))

# And the stochasticity that a deterministic translator cannot offer.
for f in sorted(glob.glob(f'{FINAL_DIR}/figures/synth/grids/variants_*_01.png')):
    print(f'=== {f.split("/")[-1]} ===')
    display(ShowImage(filename=f))


## 15. Snapshot the results

Everything already lives on Drive (`results_sweep/`, `results_final/`), so this
is not a rescue copy. It freezes a **timestamped snapshot** of the tables,
figures and chosen scales that the report is built from, so a later re-run
can't silently change the numbers you quoted. The generated images stay in
`results_final/` and are not duplicated.

```
results_<timestamp>/
  synth_metrics.csv                 <- report table 1 (the deliverable)
  roundtrip_<builder>_metrics.csv   <- report tables 2-3 (one per builder)
  *_per_image.csv                   <- per-image values, for error bars or histograms
  figures/<experiment>/grids/       <- comparison_NN.png, variants_<model>_NN.png
  figures/<experiment>/panels/      <- every cell as its own file, for hand layout
  figures/<experiment>/index.csv    <- panel dir -> source image
  sweep/<model>_sweep.csv           <- scale-sweep tables (appendix)
  sweep/chosen_scales.json          <- the rule's choice and reasoning per model
```

`docs/methods.md` in the repo explains every metric, the analytic operator and
the scale-selection rule, with formulas, for writing this up.


In [ ]:
import datetime

stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{DRIVE}/results_{stamp}'
!mkdir -p {dest}/sweep
!cp {FINAL_DIR}/*.csv {dest}/ 2>/dev/null
!cp -r {FINAL_DIR}/figures {dest}/ 2>/dev/null
!cp {SWEEP_DIR}/*_sweep.csv {SWEEP_DIR}/chosen_scales.json {dest}/sweep/ 2>/dev/null

print('snapshot saved to', dest)
!du -sh {dest}
!df -h /content/drive | tail -1
print('\nRemember to disconnect the runtime if you are done '
      '— an idle GPU still burns compute units.')
